# (노트) 루나랜더

- toc:true
- branch: master
- badges: true
- comments: true
- author: 신록예찬
- hide: false
- categories: [딥러닝]

In [1]:
#!pip install gym[Box_2D] Box2D box2d-py gym[box2d]

In [2]:
import gym
#from gym.wrappers import Monitor
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt
import base64, io

import numpy as np
from collections import deque, namedtuple

# For visualization
from gym.wrappers.monitoring import video_recorder
from IPython.display import HTML
from IPython import display 
import glob

In [3]:
env = gym.make('LunarLander-v2',new_step_api=True)

In [4]:
class ReplayBuffer:
    def __init__(self,env):
        # 초기화 
        self.env = env
        self.states = deque(maxlen=100000)  
        self.actions = deque(maxlen=100000)  
        self.rewards = deque(maxlen=100000)  
        self.next_states = deque(maxlen=100000)  
        self.dones = deque(maxlen=100000)
        self.n = 0 
        
        # random action으로 게임 진행 -> 저장 
        state1 = self.env.reset()
        for t in range(1500):
            action = self.env.action_space.sample() 
            state2,reward, done, _, _ = env.step(action) 
            self.save(state1,action,reward,state2,done) 
            state1 = state2 
            if done:
                break
                
    def save(self, state, action, reward, next_state, done):
        self.states.append(state.tolist()) 
        self.actions.append(action)
        self.rewards.append(reward)
        self.next_states.append(next_state.tolist())
        self.dones.append(done)
        self.n = self.n + 1 
    
    def sample(self):
        _index = np.random.choice(self.n,128) # 128 is batch_size 
        states = torch.tensor(self.states)[_index]
        actions = torch.tensor(self.actions)[_index]
        rewards = torch.tensor(self.rewards)[_index]
        next_states = torch.tensor(self.next_states)[_index]
        dones = torch.tensor(self.dones)[_index]
        
        actions = torch.LongTensor(actions).reshape(-1,1)
        rewards = rewards.reshape(-1,1)
        dones = dones.to(torch.float).reshape(-1,1)
        return (states, actions, rewards, next_states, dones)
    
    def __len__(self):
        return self.n

In [5]:
class Agent:
    def __init__(self,env):
        self.env = env 
        self.net = torch.nn.Sequential(
            torch.nn.Linear(in_features=8, out_features=128),
            torch.nn.ReLU(),
            torch.nn.Linear(in_features=128, out_features=64),
            torch.nn.ReLU(),
            torch.nn.Linear(in_features=64, out_features=32),
            torch.nn.ReLU(),
            torch.nn.Linear(in_features=32, out_features=4)
        )     
        self.optimizer = optim.Adam(self.net.parameters(), lr=0.0001)
        self.replaybuffer = ReplayBuffer(env)
        self.t_step = 0
    
    def step(self, state, action, reward, next_state, done):
        self.replaybuffer.save(state, action, reward, next_state, done)
        states, actions, rewards, next_states, dones = self.replaybuffer.sample()
        qvalue = self.net(next_states).detach().max(1)[0].reshape(-1,1) 
        y = rewards + 0.99 * qvalue * (1 - dones)
        yhat = self.net(states).gather(1, actions)
        loss = torch.mean((y-yhat)**2)
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()            
            
    def act(self, state, eps=0.):
        state = torch.from_numpy(state).float().unsqueeze(0)
        self.net.eval()
        with torch.no_grad():
            action_values = self.net(state)
        self.net.train()
        if random.random() > eps:
            return np.argmax(action_values.cpu().data.numpy())
        else:
            return random.choice(np.arange(4))

In [6]:
n_episodes=2000
max_t=1000
eps_start=1
eps_end=0.01
eps_decay=0.995

In [7]:
agent = Agent(env)

In [8]:
scores = []                        # list containing scores from each episode
scores_window = deque(maxlen=100)  # last 100 scores
eps = eps_start                    # initialize epsilon
for i_episode in range(1, n_episodes+1):
    state = env.reset()
    score = 0
    for t in range(max_t):
        ############################### TODO: YOUR CODE BELOW ##############################
        ## STEP1: action을 agent로부터 뽑습니다.
        action = agent.act(state, eps)
        
        ## STEP2: env.step()을 사용해서 원하는 정보를 뽑습니다.
        next_state, reward, done, _, _ = env.step(action)
        #reward = reward - 1/20*np.abs(t/50-15)
        
        ## STEP3: 뽑아낸 정보를 바탕으로 agent를 학습시킵니다.
        agent.step(state, action, reward, next_state, done)
        
        ## STEP4: 다음 상태로 갱신시킵니다
        state = next_state
        
        ## STEP5: score를 정의해주세요.
        score += reward
        
        ################################# END OF YOUR CODE #################################
        if done:
            break 
    scores_window.append(score)       # save most recent score
    scores.append(score)              # save most recent score
    eps = max(eps_end, eps_decay*eps) # decrease epsilon
    print('\rEpisode {}\tAverage Score: {:.2f}'.format(i_episode, np.mean(scores_window)), end="")
    if i_episode % 100 == 0:
        print('\rEpisode {}\tAverage Score: {:.2f}'.format(i_episode, np.mean(scores_window)))
    if np.mean(scores_window)>=200.0:
        print('\nEnvironment solved in {:d} episodes!\tAverage Score: {:.2f}'.format(i_episode-100, np.mean(scores_window)))
        torch.save(agent.net.state_dict(), 'checkpoint.pth')
        break

Episode 100	Average Score: -124.23
Episode 172	Average Score: -79.854

KeyboardInterrupt: 

In [ ]:
# plot the scores
fig = plt.figure()
ax = fig.add_subplot(111)
plt.plot(np.arange(len(scores)), scores)
plt.ylabel('Score')
plt.xlabel('Episode #')
plt.show()